In [4]:
ALL_METHODS = [
    "results___NPBIR___EST_g__EST_l",  # OG
    "results___NPBIR____GT_g__EST_l",
    "results___NPBIR___EST_g___GT_l",
    "results___NPBIR__VGGT_g__EST_l",
    "results___NPBIR__VGGT_g___GT_l",
]


METHOD=ALL_METHODS[2]
NPBIR="./DigitalTwinCatalog/neural_pbir"
CHKPT=f"./{METHOD}/stanford_orb"
SORB = "/home/ahc/Datasets/Stanford-ORB"
SORB_HDR = f"{SORB}/blender_HDR"
SORB_GT = f"{SORB}/ground_truth"
SCENE = "teapot_scene001" # "baking_scene001"

In [3]:
# run once per new scene within the dataset itself (regardless of method, etc.)
# !python DigitalTwinCatalog/neural_pbir/scripts/preprocess/stanford_orb.py /home/ahc/Datasets/Stanford-ORB/blender_HDR/{SCENE}

# !python {NPBIR}/neural_surface_recon/run_template.py --template {NPBIR}/neural_surface_recon/configs/template_stanford_orb.py --savemem {SORB_HDR}/{SCENE}/ --mesh_init_path={SORB_GT}/{SCENE}/mesh_blender/mesh.obj
# !mv results {METHOD}


# !python {NPBIR}/neural_distillation/run.py {CHKPT}/{SCENE}/

# # produce an average GT envmap from test views
# !python average_envmaps.py {SORB_GT}/{SCENE}/env_map  {SORB_HDR}/{SCENE}/transforms_test.json --output {SORB_GT}/{SCENE}/env_map/avg.exr
!python {NPBIR}/pbir/run.py {NPBIR}/pbir/configs/template {CHKPT}/{SCENE}/ --gt_envmap_path {SORB_GT}/{SCENE}/env_map/avg.exr

!python {NPBIR}/scripts/stanford_orb/render_geo.py {SORB_HDR}/{SCENE}/cameras.json {CHKPT}/{SCENE}/pbir/mesh.obj

!python {NPBIR}/scripts/stanford_orb/postproc_envmap_our.py "{CHKPT}/{SCENE}/pbir/envmap.exr"

# NOVEL VIEW SYNTHESIS
CAM=f"{SORB_HDR}/{SCENE}/cameras.json"
CKPT_GEO=f"{CHKPT}/{SCENE}/pbir/mesh.obj"
CKPT_ALBEDO=f"{CHKPT}/{SCENE}/pbir/diffuse.exr"
CKPT_ROUGH=f"{CHKPT}/{SCENE}/pbir/roughness.exr"
CKPT_ENV=f"{CHKPT}/{SCENE}/pbir/envmap_for_blender.exr"
!python {NPBIR}/scripts/relit/relit.py {CKPT_GEO} {CKPT_ALBEDO} {CKPT_ROUGH} {CAM} --lgt_paths {CKPT_ENV} --render_exr --with_bg

# RELIGHTING
!python {NPBIR}/scripts/stanford_orb/relit_nl.py --ckptroot {CHKPT} --dataroot {SORB}


!python {NPBIR}/scripts/stanford_orb/eval_preprocess.py --data_dir {SORB}/  --ckpt_dir {CHKPT}/
!cd Stanford-ORB && PYTHONPATH=. python scripts/test.py --input-path ../{CHKPT}/eval_inputs_pbir.json --output-path ../{CHKPT}/eval_outputs_pbir.json --scenes example


/home/ahc/miniconda3/envs/metrology_ir/lib/python3.11/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Running stage microfacet_naive-envmap_sg...
/home/ahc/miniconda3/envs/metrology_ir/lib/python3.11/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Starting the optimization...
 10%|████▎                                     | 52/500 [00:18<03:17,  2.27it/s]Traceback (most recent call last):
  File "/home/ahc/Docu

  aligned 0000.exr
  aligned 0007.exr
  aligned 0014.exr
  aligned 0021.exr
  aligned 0028.exr
  aligned 0035.exr
  aligned 0042.exr
  aligned 0049.exr
  aligned 0056.exr
  aligned 0063.exr
Averaged 10 envmap(s) (512x1024) -> /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/env_map/avg.exr
